In [1]:
import mne
import pickle
import hashlib
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import paper_style  # two-column paper style for every figure (sizes/font in paper_style.py)
from paper_style import COL_W, TEXT_W, CAT_COLORS, clean_axes, enforce_font_sizes, subject_id

from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

from models.EEGNet import EEGNet, ConstrainedConv2d, ConstrainedLinear, train_eegnet
from models.GATGRU import MultiBranchGATGRUTransformer, SpatialBranch, FrequencyBranch, TemporalBranch, GATLayer, train_gat_gru_transformer, channel_importance, band_importance, gat_attention, plv_report
from scripts.dataset_manager import read_raw_dataset

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\nicol\_netrc.
wandb: Currently logged in as: nbaeriswyl (nbaeriswyl-eth-z-rich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Feature extraction

In [2]:
EPOCH_DATASET_NAME = pathlib.Path('data/epoch_dataset')
EPOCH_DATASET_LIGHT_NAME = pathlib.Path('data/epoch_dataset_light')
EPOCH_FILTERED_DATASET_NAME = pathlib.Path('data/epoch_dataset_filtered')
EPOCH_ROBUST_DATASET_NAME = pathlib.Path('data/epoch_dataset_robust')
EPOCH_ROBUST_LIGHT_DATASET_NAME = pathlib.Path('data/epoch_dataset_robust_light')

ACTIVE_DATASET = EPOCH_FILTERED_DATASET_NAME
# ACTIVE_DATASET = EPOCH_DATASET_LIGHT_NAME
# ACTIVE_DATASET = EPOCH_ROBUST_DATASET_NAME

def merge_epochs_dict(epochs_dict):
    """Merges the dict of epochs into a single epoch object, preserving subject and session identification in the metadata.
    The epochs in `epochs_dict` get the subject/session metadata columns added in place (no per-session copy, which
    would briefly hold the whole dataset a third time) -- the dict is meant to be discarded after merging."""
    all_epochs = []

    for subj, sessions in epochs_dict.items():
        for sess, ep in sessions.items():
            # build/extend metadata with subject + session
            n = len(ep)
            extra = pd.DataFrame(
                {"subject": [subj] * n, "session": [sess] * n},
                index=ep.metadata.index if ep.metadata is not None else range(n),
            )

            if ep.metadata is None:
                ep.metadata = extra
            else:
                # avoid clobbering if these columns already exist
                meta = ep.metadata.copy()
                meta["subject"] = subj
                meta["session"] = sess
                ep.metadata = meta

            all_epochs.append(ep)

    merged = mne.concatenate_epochs(all_epochs)
    return merged

def query_epochs(epochs, query):
    # indexing already returns a new Epochs object holding a copy of just the selected
    # epochs, so no epochs.copy() of the full dataset first
    keep_idx = epochs.metadata.query(query).index
    mask = epochs.metadata.index.isin(keep_idx)
    return epochs[mask]

def drop_train_overlapping_test(X_train, y_train, X_test):
    """Removes all train epochs whose window overlaps any test epoch. Overlaps are only recorded from the
    overlapping (Step-3) side in `overlap_neighbor_ids` (as `epoch_id`s within the same subject/session),
    which is sufficient as long as the test set only holds non-overlapping epochs."""
    test_md = X_test.metadata
    test_keys = set(zip(test_md.subject, test_md.session, test_md.epoch_id.astype(int)))
    train_md = X_train.metadata
    overlaps_test = np.array([
        any((subj, sess, int(n)) in test_keys for n in neighbors.split(',') if n != '')
        for subj, sess, neighbors in zip(train_md.subject, train_md.session, train_md.overlap_neighbor_ids.fillna('').astype(str))
    ], dtype=bool)
    print(f"dropping {overlaps_test.sum()} train epochs overlapping the test set")
    return X_train[~overlaps_test], y_train[~overlaps_test]


epochs_scaled = merge_epochs_dict(read_raw_dataset(ACTIVE_DATASET))
light_epochs_scaled = merge_epochs_dict(read_raw_dataset(EPOCH_DATASET_LIGHT_NAME))
robust_epochs_scaled = merge_epochs_dict(read_raw_dataset(EPOCH_ROBUST_DATASET_NAME))
robust_light_epochs_scaled = merge_epochs_dict(read_raw_dataset(EPOCH_ROBUST_LIGHT_DATASET_NAME))
epochs_scaled.load_data().apply_function(lambda x: (x - x.mean()) / (x.std() + 1e-12), channel_wise=True)
light_epochs_scaled.load_data().apply_function(lambda x: (x - x.mean()) / (x.std() + 1e-12), channel_wise=True)
robust_light_epochs_scaled.load_data().apply_function(lambda x: (x - x.mean()) / (x.std() + 1e-12), channel_wise=True)
robust_epochs_scaled.load_data().apply_function(lambda x: (x - x.mean()) / (x.std() + 1e-12), channel_wise=True)

# stable per-row id on every parent dataset, carried along by every subset queried from it below --
# lets the random forest look its precomputed features up in a FeatureStore of the parent
# (see models/EEGForest.py) for any train/test split of any of these subsets
for _parent in (epochs_scaled, light_epochs_scaled, robust_epochs_scaled, robust_light_epochs_scaled):
    _parent.metadata = _parent.metadata.assign(row_id=np.arange(len(_parent)))

filtered_light_epochs_scaled = query_epochs(light_epochs_scaled, "((EventType == 'PinchLog') & (PinchType == 'PINCH')) | (EventType == 'NoPinchLog')")
filtered_robust_light_epochs_scaled = query_epochs(robust_light_epochs_scaled, "((EventType == 'PinchLog') & (PinchType == 'PINCH')) | (EventType == 'NoPinchLog')")
filtered_robust_epochs_scaled = query_epochs(robust_epochs_scaled, "((EventType == 'PinchLog') & (PinchType == 'PINCH')) | (EventType == 'NoPinchLog')")

# normal data
filtered_epochs_scaled = query_epochs(epochs_scaled, "((EventType == 'PinchLog') & (PinchType == 'PINCH')) | ((EventType == 'NoPinchLog') & (PinchType == 'Empty'))")
robust_filtered_epochs_scaled = query_epochs(robust_epochs_scaled, "((EventType == 'PinchLog') & (PinchType == 'PINCH')) | ((EventType == 'NoPinchLog') & (PinchType == 'Empty'))")

# GAZE as hard negatives
filtered_epochs_gaze_scaled = query_epochs(epochs_scaled, "((EventType == 'PinchLog') & ((PinchType == 'PINCH') | (PinchType == 'GAZE'))) | ((EventType == 'NoPinchLog') & (PinchType == 'Empty'))")
filtered_epochs_gaze_scaled.metadata.loc[:, 'OriginalEventType'] = filtered_epochs_gaze_scaled.metadata.loc[:, 'EventType']
filtered_epochs_gaze_scaled.metadata.loc[filtered_epochs_gaze_scaled.metadata.PinchType == 'GAZE', 'EventType'] = "NoPinchLog"
robust_filtered_epochs_gaze_scaled = query_epochs(robust_epochs_scaled, "((EventType == 'PinchLog') & ((PinchType == 'PINCH') | (PinchType == 'GAZE'))) | ((EventType == 'NoPinchLog') & (PinchType == 'Empty'))")
robust_filtered_epochs_gaze_scaled.metadata.loc[:, 'OriginalEventType'] = robust_filtered_epochs_gaze_scaled.metadata.loc[:, 'EventType']
robust_filtered_epochs_gaze_scaled.metadata.loc[robust_filtered_epochs_gaze_scaled.metadata.PinchType == 'GAZE', 'EventType'] = "NoPinchLog"

# PINCH_WIZARD_OF_OZ as pinches
filtered_epochs_hinted_scaled = query_epochs(epochs_scaled, "((EventType == 'PinchLog') & ((PinchType == 'PINCH') | (PinchType == 'PINCH_WIZARD_OF_OZ'))) | ((EventType == 'NoPinchLog') & (PinchType == 'Empty'))")
filtered_epochs_hinted_scaled.metadata.loc[:, 'OriginalPinchType'] = filtered_epochs_hinted_scaled.metadata.loc[:, 'PinchType']
filtered_epochs_hinted_scaled.metadata.loc[filtered_epochs_hinted_scaled.metadata.PinchType == 'PINCH_WIZARD_OF_OZ', 'PinchType'] = "PINCH"
robust_filtered_epochs_hinted_scaled = query_epochs(robust_epochs_scaled, "((EventType == 'PinchLog') & ((PinchType == 'PINCH') | (PinchType == 'PINCH_WIZARD_OF_OZ'))) | ((EventType == 'NoPinchLog') & (PinchType == 'Empty'))")
robust_filtered_epochs_hinted_scaled.metadata.loc[:, 'OriginalPinchType'] = robust_filtered_epochs_hinted_scaled.metadata.loc[:, 'PinchType']
robust_filtered_epochs_hinted_scaled.metadata.loc[robust_filtered_epochs_hinted_scaled.metadata.PinchType == 'PINCH_WIZARD_OF_OZ', 'PinchType'] = "PINCH"

# PINCH_IGNORED as pinches
filtered_epochs_ignored_scaled = query_epochs(epochs_scaled, "((EventType == 'PinchLog') & ((PinchType == 'PINCH') | (PinchType == 'PINCH_IGNORED'))) |( (EventType == 'NoPinchLog') & (PinchType == 'Empty'))")
filtered_epochs_ignored_scaled.metadata.loc[:, 'OriginalPinchType'] = filtered_epochs_ignored_scaled.metadata.loc[:, 'PinchType']
filtered_epochs_ignored_scaled.metadata.loc[filtered_epochs_ignored_scaled.metadata.PinchType == 'PINCH_IGNORED', 'PinchType'] = "PINCH"
robust_filtered_epochs_ignored_scaled = query_epochs(robust_epochs_scaled, "((EventType == 'PinchLog') & ((PinchType == 'PINCH') | (PinchType == 'PINCH_IGNORED'))) |( (EventType == 'NoPinchLog') & (PinchType == 'Empty'))")
robust_filtered_epochs_ignored_scaled.metadata.loc[:, 'OriginalPinchType'] = robust_filtered_epochs_ignored_scaled.metadata.loc[:, 'PinchType']
robust_filtered_epochs_ignored_scaled.metadata.loc[robust_filtered_epochs_ignored_scaled.metadata.PinchType == 'PINCH_IGNORED', 'PinchType'] = "PINCH"

# FINGER_MOVEMENT as negatives
filtered_epochs_fingermovement_scaled = query_epochs(epochs_scaled, "((EventType == 'PinchLog') & ((PinchType == 'PINCH') | (PinchType == 'PINCH_IGNORED'))) | (EventType == 'NoPinchLog')")
robust_filtered_epochs_fingermovement_scaled = query_epochs(robust_epochs_scaled, "((EventType == 'PinchLog') & ((PinchType == 'PINCH') | (PinchType == 'PINCH_IGNORED'))) | (EventType == 'NoPinchLog')")


Reading C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_filtered\subject_21\session_1\session-epo.fif ...
    Found the data of interest:
        t =   -1700.00 ...     300.00 ms
        0 CTF compensation matrices available
Adding metadata with 52 columns
290 matching events found
No baseline correction applied
0 projection items activated
Reading C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_filtered\subject_21\session_2\session-epo.fif ...
    Found the data of interest:
        t =   -1700.00 ...     300.00 ms
        0 CTF compensation matrices available
Adding metadata with 52 columns
128 matching events found
No baseline correction applied
0 projection items activated
Reading C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_filtered\subject_22\session_1\session-epo.fif ...
    Found the data of interest:
        t =   -1700.00 ...     300.00 ms
        0 CTF compensation matrices available
Adding metadata with 52 columns
148 matching events foun

C:\Users\nicol\AppData\Local\Temp\ipykernel_6816\11890598.py:39: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  merged = mne.concatenate_epochs(all_epochs)


Adding metadata with 52 columns
1310 matching events found
No baseline correction applied
0 projection items activated
Reading C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_light\subject_21\session_2\session-epo.fif ...
    Found the data of interest:
        t =   -1700.00 ...     300.00 ms
        0 CTF compensation matrices available
Adding metadata with 52 columns
683 matching events found
No baseline correction applied
0 projection items activated
Reading C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_light\subject_22\session_1\session-epo.fif ...
    Found the data of interest:
        t =   -1700.00 ...     300.00 ms
        0 CTF compensation matrices available
Adding metadata with 52 columns
1894 matching events found
No baseline correction applied
0 projection items activated
Reading C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_light\subject_22\session_2\session-epo.fif ...
    Found the data of interest:
        t =   -1700.00 ...     3

C:\Users\nicol\AppData\Local\Temp\ipykernel_6816\11890598.py:39: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  merged = mne.concatenate_epochs(all_epochs)


Adding metadata with 54 columns
66159 matching events found
No baseline correction applied
Reading C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_robust_filtered\subject_21\session_1\session-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    2000.00 ms
        0 CTF compensation matrices available
Adding metadata with 59 columns
1248 matching events found
No baseline correction applied
0 projection items activated
Reading C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_robust_filtered\subject_21\session_2\session-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    2000.00 ms
        0 CTF compensation matrices available
Adding metadata with 59 columns
620 matching events found
No baseline correction applied
0 projection items activated
Reading C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_robust_filtered\subject_22\session_1\session-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    

C:\Users\nicol\AppData\Local\Temp\ipykernel_6816\11890598.py:39: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  merged = mne.concatenate_epochs(all_epochs)


Adding metadata with 59 columns
28584 matching events found
No baseline correction applied
Reading C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_robust_light\subject_21\session_1\session-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    2000.00 ms
        0 CTF compensation matrices available
Adding metadata with 59 columns
1430 matching events found
No baseline correction applied
0 projection items activated
Reading C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_robust_light\subject_21\session_2\session-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    2000.00 ms
        0 CTF compensation matrices available
Adding metadata with 59 columns
638 matching events found
No baseline correction applied
0 projection items activated
Reading C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_robust_light\subject_22\session_1\session-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    2000.00 m

C:\Users\nicol\AppData\Local\Temp\ipykernel_6816\11890598.py:39: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  merged = mne.concatenate_epochs(all_epochs)


Adding metadata with 59 columns
33230 matching events found
No baseline correction applied
Replacing existing metadata with 55 columns
Replacing existing metadata with 55 columns
Replacing existing metadata with 60 columns
Replacing existing metadata with 60 columns


### Result caching

Model-training runs below are expensive, so their results are cached on disk in `RESULTS_DIR`, keyed by a run name and annotated with `DATA_VERSION` - a fingerprint of the input dataset (`EPOCH_FILTERED_DATASET_NAME`, the directory `read_raw_dataset` above loads from). If a cached result's `data_version` matches the current one, it is loaded instead of retraining. Set `FORCE_RECOMPUTE_RESULTS = True` (or pass `force=True` to `run_cached`) to ignore the cache and recompute everything.

In [3]:
RESULTS_DIR = pathlib.Path('data/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FORCE_RECOMPUTE_RESULTS = False  # set True to ignore cached results and recompute everything


def compute_data_version(dataset_path: pathlib.Path) -> str:
    """Fingerprints a dataset directory so cached run results can be invalidated when the
    underlying input data changes. Uses (relative path, size, mtime) per file rather than
    hashing file contents, since the dataset is hundreds of MB."""
    dataset_path = pathlib.Path(dataset_path)
    entries = []
    for f in sorted(dataset_path.rglob('*')):
        if f.is_file():
            stat = f.stat()
            entries.append(f"{f.relative_to(dataset_path).as_posix()}:{stat.st_size}:{int(stat.st_mtime_ns)}")
    return hashlib.sha256("\n".join(entries).encode()).hexdigest()[:16]


DATA_VERSION = compute_data_version(EPOCH_FILTERED_DATASET_NAME)


def model_data_version(dataset_path: pathlib.Path, decim: int = 1) -> str:
    """data_version for a model's cached results: the fingerprint of the dataset the run
    actually used, plus the decimation the model saw -- so results trained at another
    sampling rate are never reloaded."""
    return f"{compute_data_version(dataset_path)}-decim{decim}"


def run_cached(name: str, compute_fn, data_version: str = None, force: bool = None):
    """Runs `compute_fn()` and caches the result under `name` in RESULTS_DIR, annotated with
    `data_version` (defaults to the global DATA_VERSION). On a later call with the same name
    and data_version, the cached result is loaded instead of recomputing. Pass force=True (or
    set FORCE_RECOMPUTE_RESULTS) to always recompute and overwrite the cache."""
    data_version = DATA_VERSION if data_version is None else data_version
    force = FORCE_RECOMPUTE_RESULTS if force is None else force
    cache_path = RESULTS_DIR / f"{name}.pkl"

    if not force and cache_path.exists():
        with open(cache_path, "rb") as fh:
            cached = pickle.load(fh)
        if cached.get("data_version") == data_version:
            print(f"[{name}] loaded cached result (data_version={data_version})")
            return cached["result"]
        print(f"[{name}] cached data_version={cached.get('data_version')} != {data_version}, recomputing")
    else:
        print(f"[{name}] {'forced recompute' if force else 'no cache found'}, running...")

    result = compute_fn()
    with open(cache_path, "wb") as fh:
        pickle.dump({"data_version": data_version, "result": result}, fh)
    return result


# Model training & inference

## Random Forest

Input:

In [4]:
from contextlib import contextmanager

from models.EEGForest import train_eegforest, build_feature_store, FEATURE_STORE_VERSION


@contextmanager
def hidden_event_timing(epochs):
    """Temporarily blanks `t_offset` in `epochs.metadata` (restored afterwards, no data copy).

    EEGForest anchors all its feature windows on `t_offset` (see `_ref_times` in
    models/EEGForest.py). On the robust dataset `t_offset` is the true pinch position for
    PinchLog epochs and NaN for every NoPinchLog epoch, so it encodes the label -- and it was
    read from the *test* epochs' metadata at prediction time, leaking the label into the test
    features. With `t_offset` all-NaN every epoch is anchored at its midpoint, identically for
    both classes, so the RF sees exactly what EEGNet sees: the raw jittered windows."""
    original = epochs.metadata
    epochs.metadata = original.assign(t_offset=np.nan)
    try:
        yield epochs
    finally:
        epochs.metadata = original

# filtered_epochs_scaled etc. are locked to the pinch/no-pinch event at t=0 and only span
# T_MIN=-1.7s to T_MAX=0.3s (see 11_preprocessing.ipynb) -- much shorter than the post-movement
# windows dsi24_features.py's feature functions used to default to (designed for slower-paced
# MI paradigms with ~2.5s after the event). EEGForest's feature functions now derive their
# windows from each epochs object's own tmin/tmax instead, so no feature_kwargs/pipeline_kwargs
# override is needed here anymore.

# Every label-free part of the RF's features (handcrafted features, band-pass filtering,
# per-epoch covariances) is computed once per parent dataset into a FeatureStore. All runs below
# -- every LOSO fold of every variant, which are only row subsets/relabelings of the same parent --
# then just look their rows up (via the `row_id` metadata column) and refit the supervised spatial
# filters + the forest. Cached like the results; FEATURE_STORE_VERSION in the version tag makes a
# change to the feature code rebuild it.
# versioned on the robust dataset itself (DATA_VERSION only tracks epoch_dataset_filtered), plus a
# tag so the earlier results -- computed with the label-leaking t_offset -- are never reloaded
RF_ROBUST_DATA_VERSION = compute_data_version(EPOCH_ROBUST_DATASET_NAME) + "-no-t_offset"
rf_store = run_cached("rf_feature_store", lambda: build_feature_store(epochs_scaled, verbose=True),
                      data_version=f"{compute_data_version(ACTIVE_DATASET)}-fs{FEATURE_STORE_VERSION}")
with hidden_event_timing(robust_epochs_scaled) as _robust_parent:
    rf_robust_store = run_cached("rf_robust_feature_store", lambda: build_feature_store(_robust_parent, verbose=True),
                                 data_version=f"{RF_ROBUST_DATA_VERSION}-fs{FEATURE_STORE_VERSION}")

def rf_general(data, fix_test_set=False, store=None):
    X_train_idx, X_test_idx, y_train, y_test = train_test_split(np.arange(len(data)), data.metadata.EventType, test_size=.2)
    X_train, X_test = data[X_train_idx], data[X_test_idx]

    if fix_test_set:
        y_test = y_test[~X_test.metadata.is_overlapping]
        X_test = X_test[~X_test.metadata.is_overlapping]
        X_train, y_train = drop_train_overlapping_test(X_train, y_train, X_test)

    return train_eegforest(X_train, y_train, X_test, y_test, store=store, run_name="rf-general", log_wandb=False), X_test.metadata

def rf_cross_subject(data, leave_out='', ignore='', augment=False, fix_test_set=False, store=None):
    train_subjects, test_subjects = train_test_split(data.metadata["subject"].unique().tolist())
    if leave_out != '':
        train_subjects = data.metadata["subject"].unique().tolist()
        train_subjects.remove(leave_out)
        test_subjects = [leave_out]
    if ignore != '':
        train_subjects.remove(ignore)

    print(f"{len(train_subjects)} train subjects, {len(test_subjects)} test subjects")

    train_epochs = query_epochs(data, f"subject in {train_subjects}")
    if fix_test_set:
        test_epochs = query_epochs(data, f"(subject in {test_subjects}) & ~is_overlapping")
    else:
        test_epochs = query_epochs(data, f"subject in {test_subjects}")

    return train_eegforest(train_epochs, train_epochs.metadata.EventType, test_epochs, test_epochs.metadata.EventType,
                            store=store, log_wandb=False, augment=augment), test_epochs.metadata

def rf_cross_session(data, fix_test_set=False, store=None):
    train_epochs = query_epochs(data, f"session == 'session_1'")
    if fix_test_set:
        test_epochs = query_epochs(data, f"(session == 'session_2') & ~is_overlapping")
    else:
        test_epochs = query_epochs(data, f"session == 'session_2'")

    return train_eegforest(train_epochs, train_epochs.metadata.EventType, test_epochs, test_epochs.metadata.EventType, store=store, log_wandb=False), test_epochs.metadata

def rf_leave_one_out(data, augment=False, fix_test_set=False, store=None):
    results = {}
    for subject in data.metadata["subject"].unique().tolist():
        result, meta = rf_cross_subject(data, leave_out=subject, augment=augment, fix_test_set=fix_test_set, store=store)
        results[subject] = {
            "result": result,
            "test_metadata": meta
        }
    return results

def rf_leave_one_out_and_test_others(data, store=None):
    results = {}
    subjects = data.metadata["subject"].unique().tolist()
    for s in subjects:
        results[s] = {}
        for t in subjects:
            if s == t:
                continue
            result, meta = rf_cross_subject(data, leave_out=s, ignore=t, store=store)
            results[s][t] = {
                "result": result,
                "test_metadata": meta
            }
    return results

# NB: result variables are named *_run so they don't shadow the rf_general/rf_cross_subject
# function definitions above -- rf_leave_one_out still needs to call rf_cross_subject later.
rf_general_run, rf_general_meta = run_cached("rf_general_run", lambda: rf_general(filtered_epochs_scaled, store=rf_store))
rf_cross_session_run, results_cross_session_meta = run_cached("rf_cross_session_run", lambda: rf_cross_session(filtered_epochs_scaled, store=rf_store))
rf_leave_one_out_run = run_cached("rf_leave_one_out_run", lambda: rf_leave_one_out(filtered_epochs_scaled, store=rf_store))
rf_leave_one_out_gaze_run = run_cached("rf_leave_one_out_gaze_run", lambda: rf_leave_one_out(filtered_epochs_gaze_scaled, store=rf_store))
rf_leave_one_out_hinted_run = run_cached("rf_leave_one_out_hinted_run", lambda: rf_leave_one_out(filtered_epochs_hinted_scaled, store=rf_store))
rf_leave_one_out_ignored_run = run_cached("rf_leave_one_out_ignored_run", lambda: rf_leave_one_out(filtered_epochs_ignored_scaled, store=rf_store))
rf_leave_one_out_augmented_run = run_cached("rf_leave_one_out_augmented_run", lambda: rf_leave_one_out(filtered_epochs_scaled, augment=True, store=rf_store))
rf_leave_one_out_finger_run = run_cached("rf_leave_one_out_finger_run", lambda: rf_leave_one_out(filtered_epochs_fingermovement_scaled, store=rf_store))

with hidden_event_timing(filtered_robust_epochs_scaled) as rf_robust_input:
    rf_robust_general_run, rf_robust_general_meta = run_cached("rf_robust_general_run", lambda: rf_general(rf_robust_input, fix_test_set=True, store=rf_robust_store),data_version=RF_ROBUST_DATA_VERSION,)
    rf_robust_leave_one_out_run = run_cached("rf_robust_leave_one_out_run", lambda: rf_leave_one_out(rf_robust_input, fix_test_set=True, store=rf_robust_store),data_version=RF_ROBUST_DATA_VERSION,)
    rf_robust_cross_session_run, results_robust_cross_session_meta = run_cached("rf_robust_cross_session_run", lambda: rf_cross_session(rf_robust_input, fix_test_set=True, store=rf_robust_store),data_version=RF_ROBUST_DATA_VERSION,)
with hidden_event_timing(robust_filtered_epochs_gaze_scaled) as rf_robust_input:
    rf_robust_leave_one_out_gaze_run = run_cached("rf_robust_leave_one_out_gaze_run", lambda: rf_leave_one_out(rf_robust_input, fix_test_set=True, store=rf_robust_store),data_version=RF_ROBUST_DATA_VERSION)
with hidden_event_timing(robust_filtered_epochs_hinted_scaled) as rf_robust_input:
    rf_robust_leave_one_out_hinted_run = run_cached("rf_robust_leave_one_out_hinted_run", lambda: rf_leave_one_out(rf_robust_input, fix_test_set=True, store=rf_robust_store),data_version=RF_ROBUST_DATA_VERSION)
with hidden_event_timing(robust_filtered_epochs_ignored_scaled) as rf_robust_input:
    rf_robust_leave_one_out_ignored_run = run_cached("rf_robust_leave_one_out_ignored_run", lambda: rf_leave_one_out(rf_robust_input, fix_test_set=True, store=rf_robust_store),data_version=RF_ROBUST_DATA_VERSION)
with hidden_event_timing(robust_filtered_epochs_fingermovement_scaled) as rf_robust_input:
    rf_robust_leave_one_out_finger_run = run_cached("rf_robust_leave_one_out_finger_run", lambda: rf_leave_one_out(rf_robust_input, fix_test_set=True, store=rf_robust_store),data_version=RF_ROBUST_DATA_VERSION)

[rf_feature_store] cached data_version=4925275ab3bb2a4a-fs1 != 39ff0cb877b38b23-fs1, recomputing
feature store: chunk 7/7
[rf_robust_feature_store] cached data_version=1cf63b8769179382-no-t_offset-fs1 != bae66ec694fa98a2-no-t_offset-fs1, recomputing
feature store: chunk 29/29
[rf_leave_one_out_finger_run] no cache found, running...
18 train subjects, 1 test subjects
EEGForest | supervised=['fbcsp', 'riemann', 'xdawn'] | 2 classes | 4242 train / 1060 val / 367 test epochs | n_estimators=400 | cached features
trees   25/400  loss 0.2293  acc 0.999  val_loss 0.6613  val_acc 0.603
trees   50/400  loss 0.2280  acc 1.000  val_loss 0.6445  val_acc 0.647
trees   75/400  loss 0.2281  acc 1.000  val_loss 0.6441  val_acc 0.635
trees  100/400  loss 0.2271  acc 1.000  val_loss 0.6423  val_acc 0.632
trees  125/400  loss 0.2266  acc 1.000  val_loss 0.6407  val_acc 0.643
trees  150/400  loss 0.2257  acc 1.000  val_loss 0.6421  val_acc 0.653
trees  175/400  loss 0.2256  acc 1.000  val_loss 0.6400  val_

## EEGNet

This is an implementation of EEGNet according to the original paper (Lawhern et al., 2018) in PyTorch.

Input: The EEGNet takes the raw measurements as inputs.

In [5]:
# region Plotting Helpers
def _get_history_dict(history):
    """Support Keras History objects or plain dicts of lists."""
    return history.history if hasattr(history, "history") else history

def _chance_level(result):
    """
    Estimate the baseline accuracy using the majority-class baseline.
    """
    cm = np.asarray(result["confusion_matrix"])
    true_class_counts = cm.sum(axis=1)  # rows = true labels
    total = true_class_counts.sum()
    return true_class_counts.max() / total


def plot_run(result, run_name="run", save_path=None):
    """Plot loss/accuracy curves and confusion matrix for a single run."""
    hist = _get_history_dict(result["history"])
    n_epochs = len(next(iter(hist.values())))
    epochs = range(1, n_epochs + 1)

    fig, axd = plt.subplot_mosaic([["loss", "acc"], ["cm", "cm"]], figsize=(COL_W, 3.6), layout="constrained")
    axes = [axd["loss"], axd["acc"], axd["cm"]]
    fig.suptitle(
        f"{run_name} ﷿﷿﷿ test acc: {result['test_accuracy']:.3f}, "
        f"test loss: {result['test_loss']:.3f}"
    )

    # Loss curves
    ax = axes[0]
    if "loss" in hist:
        ax.plot(epochs, hist["loss"], label="train")
    if "val_loss" in hist:
        ax.plot(epochs, hist["val_loss"], label="val")
    ax.set_title("Loss")
    ax.set_xlabel("Epoch")
    ax.legend()

    # Accuracy curves
    ax = axes[1]
    acc_key = "accuracy" if "accuracy" in hist else "acc"
    val_acc_key = "val_accuracy" if "val_accuracy" in hist else "val_acc"
    if acc_key in hist:
        ax.plot(epochs, hist[acc_key], label="train")
    if val_acc_key in hist:
        ax.plot(epochs, hist[val_acc_key], label="val")

    chance = _chance_level(result)
    ax.axhline(
        chance, color="gray", linestyle="--", linewidth=1.0,
        label=f"chance ({chance:.2f})",
    )

    ax.set_title("Accuracy")
    ax.set_xlabel("Epoch")
    ax.legend()

    # Confusion matrix
    ax = axes[2]
    disp = ConfusionMatrixDisplay(
        confusion_matrix=result["confusion_matrix"],
        display_labels=result["classes"],
    )
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title("Confusion matrix")

    enforce_font_sizes(fig)
    if save_path:
        fig.savefig(save_path)
    return fig

# endregion

### Comparing cross subject vs cross session

In [6]:
# Downsample EEGNet input (take every 2nd sample)
EEGNET_DECIM = 2
eegnet_version = {d: model_data_version(d, EEGNET_DECIM) for d in
                  (ACTIVE_DATASET, EPOCH_DATASET_LIGHT_NAME, EPOCH_ROBUST_DATASET_NAME, EPOCH_ROBUST_LIGHT_DATASET_NAME)}

def classify_general(data, fix_test_set=False, save_as=None):
    X_train_idx, X_test_idx, y_train, y_test = train_test_split(np.arange(len(data)), data.metadata.EventType, test_size=.2)
    X_train, X_test = data[X_train_idx], data[X_test_idx]

    if fix_test_set:
        y_test = y_test[~X_test.metadata.is_overlapping]
        X_test = X_test[~X_test.metadata.is_overlapping]
        X_train, y_train = drop_train_overlapping_test(X_train, y_train, X_test)

    return train_eegnet(X_train, y_train, X_test, y_test, log_wandb=False, save_as=save_as, decim=EEGNET_DECIM), X_test.metadata

def classify_cross_subject(data, leave_out='', ignore='', augment=False, fix_test_set=False):
    train_subjects, test_subjects = train_test_split(data.metadata["subject"].unique().tolist())
    if leave_out != '':
        train_subjects = data.metadata["subject"].unique().tolist()
        train_subjects.remove(leave_out)
        test_subjects = [leave_out]
    if ignore != '':
        train_subjects.remove(ignore)

    print(f"{len(train_subjects)} train subjects, {len(test_subjects)} test subjects - leaving out {test_subjects}")

    train_epochs = query_epochs(data, f"subject in {train_subjects}")
    if fix_test_set:
        test_epochs = query_epochs(data, f"(subject in {test_subjects}) & ~is_overlapping")
    else:
        test_epochs = query_epochs(data, f"subject in {test_subjects}")

    return train_eegnet(train_epochs, train_epochs.metadata.EventType, test_epochs, test_epochs.metadata.EventType, log_wandb=False, n_epochs=100, augment=augment, decim=EEGNET_DECIM), test_epochs.metadata

def classify_cross_session(data, fix_test_set=False):
    train_epochs = query_epochs(data, f"session == 'session_1'")
    if fix_test_set:
        test_epochs = query_epochs(data, f"(session == 'session_2') & ~is_overlapping")
    else:
        test_epochs = query_epochs(data, f"session == 'session_2'")

    return train_eegnet(train_epochs, train_epochs.metadata.EventType, test_epochs, test_epochs.metadata.EventType, log_wandb=False, n_epochs=100, decim=EEGNET_DECIM), test_epochs.metadata

def classify_leave_one_out(data, augment=False, fix_test_set=False):
    results = {}
    for subject in data.metadata["subject"].unique().tolist():
        result, meta = classify_cross_subject(data, leave_out=subject, augment=augment, fix_test_set=fix_test_set)
        results[subject] = {
            "result": result,
            "test_metadata": meta
        }
    return results

def classify_leave_one_out_and_test_others(data):
    results = {}
    subjects = data.metadata["subject"].unique().tolist()
    for s in subjects:
        results[s] = {}
        for t in subjects:
            if s == t:
                continue
            result, meta = classify_cross_subject(data, leave_out=s, ignore=t)
            results[s][t] = {
                "result": result,
                "test_metadata": meta
            }
    return results

results_general, results_general_meta = run_cached("eegnet_general_run", lambda: classify_general(filtered_epochs_scaled), data_version=eegnet_version[ACTIVE_DATASET])
results_filtered_general, results_filtered_general_meta = run_cached("eegnet_general_run", lambda: classify_general(filtered_light_epochs_scaled, save_as="out/fixed_model.pt2"), data_version=eegnet_version[EPOCH_DATASET_LIGHT_NAME])
results_cross_session, results_cross_session_meta = run_cached("eegnet_cross_session_run", lambda: classify_cross_session(filtered_epochs_scaled), data_version=eegnet_version[ACTIVE_DATASET])
results_leave_one_out = run_cached("eegnet_leave_one_out_run", lambda: classify_leave_one_out(filtered_epochs_scaled), data_version=eegnet_version[ACTIVE_DATASET])
results_leave_one_out1 = run_cached("eegnet_leave_one_out_gaze_run", lambda: classify_leave_one_out(filtered_epochs_gaze_scaled), data_version=eegnet_version[ACTIVE_DATASET])
results_leave_one_out2 = run_cached("eegnet_leave_one_out_hinted_run", lambda: classify_leave_one_out(filtered_epochs_hinted_scaled), data_version=eegnet_version[ACTIVE_DATASET])
results_leave_one_out3 = run_cached("eegnet_leave_one_out_ignored_run", lambda: classify_leave_one_out(filtered_epochs_ignored_scaled), data_version=eegnet_version[ACTIVE_DATASET])
results_leave_one_out_augmented_run = run_cached("eegnet_leave_one_out_augmented_run", lambda: classify_leave_one_out(filtered_epochs_scaled, augment=True), data_version=eegnet_version[ACTIVE_DATASET])
eegnet_leave_one_out_finger_run = run_cached("eegnet_leave_one_out_finger_run", lambda: classify_leave_one_out(filtered_epochs_fingermovement_scaled), data_version=eegnet_version[ACTIVE_DATASET])

# same but with robust dataset
eegnet_robust_general_run, eegnet_robust_general_run_meta = run_cached("eegnet_robust_general_run", lambda: classify_general(filtered_robust_epochs_scaled, fix_test_set=True), data_version=eegnet_version[EPOCH_ROBUST_DATASET_NAME])
eegnet_robust_light_general_run, eegnet_robust_light_general_run_meta = run_cached("eegnet_robust_light_general_run", lambda: classify_general(filtered_robust_light_epochs_scaled, fix_test_set=True, save_as="out/robust_model.pt2"), data_version=eegnet_version[EPOCH_ROBUST_LIGHT_DATASET_NAME])
results_robust_cross_session, results_robust_cross_session_meta = run_cached("eegnet_robust_cross_session_run", lambda: classify_cross_session(filtered_robust_epochs_scaled, fix_test_set=True), data_version=eegnet_version[EPOCH_ROBUST_DATASET_NAME])
results_robust_leave_one_out = run_cached("eegnet_robust_leave_one_out_run", lambda: classify_leave_one_out(filtered_robust_epochs_scaled, fix_test_set=True), data_version=eegnet_version[EPOCH_ROBUST_DATASET_NAME])
results_robust_leave_one_out1 = run_cached("eegnet_robust_leave_one_out_gaze_run", lambda: classify_leave_one_out(robust_filtered_epochs_gaze_scaled, fix_test_set=True), data_version=eegnet_version[EPOCH_ROBUST_DATASET_NAME])
results_robust_leave_one_out2 = run_cached("eegnet_robust_leave_one_out_hinted_run", lambda: classify_leave_one_out(robust_filtered_epochs_hinted_scaled, fix_test_set=True), data_version=eegnet_version[EPOCH_ROBUST_DATASET_NAME])
results_robust_leave_one_out3 = run_cached("eegnet_robust_leave_one_out_ignored_run", lambda: classify_leave_one_out(robust_filtered_epochs_ignored_scaled, fix_test_set=True), data_version=eegnet_version[EPOCH_ROBUST_DATASET_NAME])
eegnet_robust_leave_one_out_run = run_cached("eegnet_robust_leave_one_out_finger_run", lambda: classify_leave_one_out(robust_filtered_epochs_fingermovement_scaled, fix_test_set=True), data_version=eegnet_version[EPOCH_ROBUST_DATASET_NAME])

[eegnet_filtered_general_run] cached data_version=4925275ab3bb2a4a != 47212345fc8d8301-decim2, recomputing


C:\Users\nicol\github\VIBE\DataAnalysis\.venv\Lib\site-packages\torch\nn\modules\conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1105.)
  return F.conv2d(


EEGNet-8,2 | 21 ch x 301 samples @ 150 Hz (decim 2) | 2 classes | 1818 trainable params | device=cuda
epoch    1/100  loss 0.6933  acc 0.528  val_loss 0.6779  val_acc 0.584
epoch   10/100  loss 0.6069  acc 0.675  val_loss 0.5961  val_acc 0.666
epoch   20/100  loss 0.5717  acc 0.698  val_loss 0.5962  val_acc 0.676
epoch   30/100  loss 0.5547  acc 0.715  val_loss 0.5851  val_acc 0.680
epoch   40/100  loss 0.5468  acc 0.710  val_loss 0.5780  val_acc 0.702
epoch   50/100  loss 0.5337  acc 0.734  val_loss 0.5693  val_acc 0.700
epoch   60/100  loss 0.5201  acc 0.739  val_loss 0.5679  val_acc 0.703
epoch   70/100  loss 0.5149  acc 0.741  val_loss 0.5687  val_acc 0.704
epoch   80/100  loss 0.5104  acc 0.737  val_loss 0.5639  val_acc 0.707
epoch   90/100  loss 0.4922  acc 0.751  val_loss 0.5532  val_acc 0.712
epoch  100/100  loss 0.4885  acc 0.754  val_loss 0.5482  val_acc 0.700

Test loss 0.5713 | test accuracy 0.6842
Confusion matrix (rows = true, cols = predicted):
[[544 218]
 [219 403]]


W1002 07:33:05.028000 6816 .venv\Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


[eegnet_leave_one_out_finger_run] no cache found, running...
18 train subjects, 1 test subjects - leaving out ['subject_21']
EEGNet-8,2 | 21 ch x 301 samples @ 150 Hz (decim 2) | 2 classes | 1818 trainable params | device=cuda
epoch    1/100  loss 0.6951  acc 0.520  val_loss 0.6838  val_acc 0.569
epoch   10/100  loss 0.6265  acc 0.648  val_loss 0.6266  val_acc 0.655
epoch   20/100  loss 0.5923  acc 0.682  val_loss 0.5937  val_acc 0.674
epoch   30/100  loss 0.5687  acc 0.698  val_loss 0.5718  val_acc 0.707
epoch   40/100  loss 0.5591  acc 0.704  val_loss 0.5702  val_acc 0.696
epoch   50/100  loss 0.5429  acc 0.721  val_loss 0.5796  val_acc 0.699
epoch   60/100  loss 0.5275  acc 0.734  val_loss 0.5778  val_acc 0.681
epoch   70/100  loss 0.5263  acc 0.730  val_loss 0.5732  val_acc 0.696
epoch   80/100  loss 0.5254  acc 0.732  val_loss 0.5749  val_acc 0.681
epoch   90/100  loss 0.5151  acc 0.741  val_loss 0.5833  val_acc 0.678
epoch  100/100  loss 0.5161  acc 0.740  val_loss 0.5883  val_ac

In [7]:
# _ = plot_run(results_general, run_name="general")
# _ = plot_run(results_cross_session, run_name="cross_session")

In [8]:
def plot_leave_one_out_run(results):
    fig, ax = plt.subplots(figsize=(COL_W, 1.9), layout="constrained")
    ax.bar([subject_id(s) for s in results], [result['result']['test_accuracy'] for result in results.values()],
           color=CAT_COLORS[0])
    ax.set_xlabel("Participant ID")
    ax.set_ylabel("Accuracy")
    ax.set_title("Leave-one-subject-out accuracy")
    clean_axes(ax)
    plt.show()

    best_subject = max(results, key=lambda k: results[k]['result']['test_accuracy'])
    worst_subject = min(results, key=lambda k: results[k]['result']['test_accuracy'])

    plot_run(results[best_subject]['result'], run_name=best_subject)
    plot_run(results[worst_subject]['result'], run_name=worst_subject)

# plot_leave_one_out_run(results_leave_one_out)

### Comparing with hard-negatives

## Multi-Branch GAT-GRU-Transformer

Wang Z and Wang Y (2025)

Input:

In [9]:
# Downsample input (take every 2nd sample).
TRANSFORMER_DECIM = 2
transformer_version = {d: model_data_version(d, TRANSFORMER_DECIM) for d in (ACTIVE_DATASET, EPOCH_ROBUST_DATASET_NAME)}

def transformer_general(data, fix_test_set=False):
    X_train_idx, X_test_idx, y_train, y_test = train_test_split(np.arange(len(data)), data.metadata.EventType, test_size=.2)
    X_train, X_test = data[X_train_idx], data[X_test_idx]

    if fix_test_set:
        y_test = y_test[~X_test.metadata.is_overlapping]
        X_test = X_test[~X_test.metadata.is_overlapping]
        X_train, y_train = drop_train_overlapping_test(X_train, y_train, X_test)

    out = train_gat_gru_transformer(X_train, y_train, X_test, y_test, sfreq=300.0, epochs=15, batch_size=32, log_wandb=False, decim=TRANSFORMER_DECIM)
    print("test accuracy:", out["test_accuracy"])
    print(out["confusion_matrix"])
    return out

def transformer_cross_subject(data, leave_out='', augment=False, fix_test_set=False):
    train_subjects, test_subjects = train_test_split(data.metadata["subject"].unique().tolist())
    if leave_out != '':
        train_subjects = data.metadata["subject"].unique().tolist()
        train_subjects.remove(leave_out)
        test_subjects = [leave_out]

    print(f"{len(train_subjects)} train subjects, {len(test_subjects)} test subjects")

    train_epochs = query_epochs(data, f"subject in {train_subjects}")
    if fix_test_set:
        test_epochs = query_epochs(data, f"(subject in {test_subjects}) & ~is_overlapping")
    else:
        test_epochs = query_epochs(data, f"subject in {test_subjects}")

    return train_gat_gru_transformer(train_epochs, train_epochs.metadata.EventType, test_epochs, test_epochs.metadata.EventType, sfreq=300.0, epochs=15, batch_size=32, log_wandb=False, lr=1e-4, augment=augment, decim=TRANSFORMER_DECIM), test_epochs.metadata

def transformer_cross_session(data, fix_test_set=False):
    train_epochs = query_epochs(data, f"session == 'session_1'")
    if fix_test_set:
        test_epochs = query_epochs(data, f"(session == 'session_2') & ~is_overlapping")
    else:
        test_epochs = query_epochs(data, f"session == 'session_2'")

    return train_gat_gru_transformer(train_epochs, train_epochs.metadata.EventType, test_epochs, test_epochs.metadata.EventType, sfreq=300.0, epochs=15, batch_size=32, log_wandb=False, lr=1e-4, decim=TRANSFORMER_DECIM), test_epochs.metadata

def transformer_leave_one_out(data, augment=False, fix_test_set=False):
    results = {}
    for subject in data.metadata["subject"].unique().tolist():
        result, metadata = transformer_cross_subject(data, leave_out=subject, augment=augment, fix_test_set=fix_test_set)
        results[subject] = {
            'result': result,
            'test_metadata': metadata
        }
    return results

transformer_general_run = run_cached("transformer_general_run", lambda: transformer_general(filtered_epochs_scaled), data_version=transformer_version[ACTIVE_DATASET])
transformer_cross_session_run = run_cached("transformer_cross_session_run", lambda: transformer_cross_session(filtered_epochs_scaled), data_version=transformer_version[ACTIVE_DATASET])
transformer_leave_one_out_run = run_cached("transformer_leave_one_out_run", lambda: transformer_leave_one_out(filtered_epochs_scaled), data_version=transformer_version[ACTIVE_DATASET])
transformer_leave_one_out_gaze_run = run_cached("transformer_leave_one_out_gaze_run", lambda: transformer_leave_one_out(filtered_epochs_gaze_scaled), data_version=transformer_version[ACTIVE_DATASET])
transformer_leave_one_out_hinted_run = run_cached("transformer_leave_one_out_hinted_run", lambda: transformer_leave_one_out(filtered_epochs_hinted_scaled), data_version=transformer_version[ACTIVE_DATASET])
transformer_leave_one_out_ignored_run = run_cached("transformer_leave_one_out_ignored_run", lambda: transformer_leave_one_out(filtered_epochs_ignored_scaled), data_version=transformer_version[ACTIVE_DATASET])
transformer_leave_one_out_augmented_run = run_cached("transformer_leave_one_out_augmented_run", lambda: transformer_leave_one_out(filtered_epochs_scaled, augment=True), data_version=transformer_version[ACTIVE_DATASET])
transformer_leave_one_out_finger_run = run_cached("transformer_leave_one_out_finger_run", lambda: transformer_leave_one_out(filtered_epochs_fingermovement_scaled), data_version=transformer_version[ACTIVE_DATASET])

transformer_robust_general_run = run_cached("transformer_robust_general_run", lambda: transformer_general(filtered_robust_epochs_scaled, fix_test_set=True), data_version=transformer_version[EPOCH_ROBUST_DATASET_NAME])
transformer_robust_cross_session_run = run_cached("transformer_robust_cross_session_run", lambda: transformer_cross_session(filtered_robust_epochs_scaled, fix_test_set=True), data_version=transformer_version[EPOCH_ROBUST_DATASET_NAME])
transformer_robust_leave_one_out_run = run_cached("transformer_robust_leave_one_out_run", lambda: transformer_leave_one_out(filtered_robust_epochs_scaled, fix_test_set=True), data_version=transformer_version[EPOCH_ROBUST_DATASET_NAME])
transformer_robust_leave_one_out_gaze_run = run_cached("transformer_robust_leave_one_out_gaze_run", lambda: transformer_leave_one_out(robust_filtered_epochs_gaze_scaled, fix_test_set=True), data_version=transformer_version[EPOCH_ROBUST_DATASET_NAME])
transformer_robust_leave_one_out_hinted_run = run_cached("transformer_robust_leave_one_out_hinted_run", lambda: transformer_leave_one_out(robust_filtered_epochs_hinted_scaled, fix_test_set=True), data_version=transformer_version[EPOCH_ROBUST_DATASET_NAME])
transformer_robust_leave_one_out_ignored_run = run_cached("transformer_robust_leave_one_out_ignored_run", lambda: transformer_leave_one_out(robust_filtered_epochs_ignored_scaled, fix_test_set=True), data_version=transformer_version[EPOCH_ROBUST_DATASET_NAME])
transformer_robust_leave_one_out_finger_run = run_cached("transformer_robust_leave_one_out_finger_run", lambda: transformer_leave_one_out(robust_filtered_epochs_fingermovement_scaled, fix_test_set=True), data_version=transformer_version[EPOCH_ROBUST_DATASET_NAME])

[transformer_leave_one_out_finger_run] no cache found, running...
18 train subjects, 1 test subjects
[model] 98,786 parameters on cuda | 21 ch x 301 samples @ 150 Hz | 2 classes | graph density 0.40
epoch    1/15 | train 0.6949/0.5390 | val 0.6682/0.5989 | 3.4s
epoch   10/15 | train 0.6379/0.6460 | val 0.6402/0.6516 | 1.7s
epoch   15/15 | train 0.6298/0.6586 | val 0.6398/0.6497 | 1.7s

test accuracy 0.6703 | macro-F1 0.6624 | best epoch 8
              precision    recall  f1-score   support

  NoPinchLog     0.7917    0.4974    0.6109       191
    PinchLog     0.6113    0.8580    0.7139       176

    accuracy                         0.6703       367
   macro avg     0.7015    0.6777    0.6624       367
weighted avg     0.7052    0.6703    0.6603       367

18 train subjects, 1 test subjects
[model] 98,786 parameters on cuda | 21 ch x 301 samples @ 150 Hz | 2 classes | graph density 0.40
epoch    1/15 | train 0.6968/0.5325 | val 0.6781/0.5537 | 2.0s
epoch   10/15 | train 0.6327/0.654

In [10]:
_ = plot_run(transformer_general_run, run_name="Transformer") 

NameError: name 'transformer_general_run' is not defined

In [ ]:
plot_leave_one_out_run(transformer_leave_one_out_run)
# transformer_leave_one_out_run['result']

### Model interpretability (paper section 2.5) ﷿﷿﷿ best leave-one-out subject

Runs the interpretability helpers defined in the Multi-Branch GAT-GRU-Transformer cell (`channel_importance`, `band_importance`, `gat_attention`, `plv_report`) on the leave-one-out fold with the highest held-out test accuracy.

In [ ]:
best_subject = max(transformer_leave_one_out_run, key=lambda k: transformer_leave_one_out_run[k]['result']['test_accuracy'])
best_result = transformer_leave_one_out_run[best_subject]['result']
model = best_result['model']
ch_names = best_result['ch_names']
classes = best_result['classes']

test_epochs = query_epochs(filtered_epochs_scaled, f"subject in ['{best_subject}']")
lut = {c: i for i, c in enumerate(classes)}
y_enc = np.array([lut[v] for v in test_epochs.metadata.EventType.values])

print(f"best subject: {best_subject} | test accuracy {best_result['test_accuracy']:.4f} | "
      f"macro-F1 {best_result['test_macro_f1']:.4f} | {len(test_epochs)} held-out trials")

In [ ]:
chan_imp = channel_importance(model, test_epochs, y_enc, ch_names, top_k=10)
plt.figure(figsize=(COL_W, 2.4))
plt.barh([c for c, _ in chan_imp[::-1]], [v for _, v in chan_imp[::-1]], color="#4C72B0")
plt.xlabel("Gradient ﷿﷿ input attribution")
plt.title(f"Channel importance ({best_subject})")
plt.tight_layout()
plt.show()

In [ ]:
band_imp = band_importance(model, test_epochs, y_enc)
plt.figure(figsize=(COL_W, 2.2))
plt.bar(list(band_imp.keys()), list(band_imp.values()), color="#55A868")
plt.ylabel("Gradient ﷿﷿ input attribution")
plt.title(f"Frequency-band importance ({best_subject})")
plt.tight_layout()
plt.show()

In [ ]:
gat_top, gat_att = gat_attention(model, test_epochs, ch_names, top_k=10)
fig, axes = plt.subplots(2, 1, figsize=(COL_W, 5.4), gridspec_kw={"height_ratios": [1, 2]})
axes[0].barh([c for c, _ in gat_top[::-1]], [v for _, v in gat_top[::-1]], color="#C44E52")
axes[0].set_title(f"Top incoming GAT attention ({best_subject})")

im = axes[1].imshow(gat_att, cmap="viridis")
axes[1].set_xticks(range(len(ch_names))); axes[1].set_xticklabels(ch_names, rotation=90)
axes[1].set_yticks(range(len(ch_names))); axes[1].set_yticklabels(ch_names)
axes[1].set_title("Mean GAT attention matrix")
fig.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
plv = plv_report(test_epochs, sfreq=300.0, ch_names=ch_names, bands=("alpha", "beta", "gamma"))
for band, pairs in plv.items():
    print(f"{band}:")
    for a, b, v in pairs[:5]:
        print(f"  {a}-{b}: {v:.3f}")

## Analysing free variables

This chapter highlights differences in classification performance from a free variable standpoint.

In [ ]:
# results_general_meta

In [ ]:
FREE_VARS = ['feedbackIsDelayed', 'hasVirtualHand', 'hasAuditoryFeedback', 'hasVisualFeedback', 'isChoiceStimuli']

def _loo_fold_metadata(payload):
    """Attaches y_pred and per-class probability columns to one leave-one-subject-out fold's test metadata."""
    result = payload["result"]
    meta = payload["test_metadata"].copy()
    meta["y_pred"] = result["y_pred"]
    classes = result["classes"]
    for i, c in enumerate(classes):
        meta[f"y_proba_{c}"] = result["y_proba"][:, i]
    meta["proba_true"] = np.where(
        meta["EventType"] == "PinchLog", meta["y_proba_PinchLog"], meta["y_proba_NoPinchLog"]
    )
    return meta

def _hardlabel_precision(metadata, v):
    """Fraction of correctly predicted labels per group, for one free variable."""
    def frac(hit_query, total_query):
        total = len(metadata.query(total_query))
        if total == 0:
            return np.nan, 0
        return len(metadata.query(hit_query)) / total, total

    pinch_true, n_true = frac(
        f"{v} == True & EventType == 'PinchLog' & y_pred == 'PinchLog'",
        f"{v} == True & EventType == 'PinchLog'",
    )
    pinch_false, n_false = frac(
        f"{v} == False & EventType == 'PinchLog' & y_pred == 'PinchLog'",
        f"{v} == False & EventType == 'PinchLog'",
    )
    no_pinch, n_no_pinch = frac(
        "EventType == 'NoPinchLog' & y_pred == 'NoPinchLog'",
        "EventType == 'NoPinchLog'",
    )
    return [pinch_true, pinch_false, no_pinch], [n_true, n_false, n_no_pinch]


def plot_free_variable_comparison_loo(results_leave_one_out, group_stat_fn, ylabel, title, agg="std"):
    """
    Aggregates a per-group statistic (computed per left-out subject via group_stat_fn) across
    leave-one-subject-out folds, and plots the across-fold mean with std/SEM error bars - this
    captures subject-to-subject variability that a single train/test split can't show.
    """
    fig, axes = plt.subplots(3, 2, figsize=(COL_W, 5.2))
    axes = axes.flatten()

    for i, v in enumerate(FREE_VARS):
        fold_values, fold_ns = [], []
        for payload in results_leave_one_out.values():
            values, ns = group_stat_fn(_loo_fold_metadata(payload), v)
            fold_values.append(values)
            fold_ns.append(ns)

        values = np.array(fold_values, dtype=float)  # (n_folds, 3)
        ns = np.nansum(np.array(fold_ns, dtype=float), axis=0).astype(int)

        means = np.nanmean(values, axis=0)
        stds = np.nanstd(values, axis=0, ddof=1)
        n_folds = np.sum(~np.isnan(values), axis=0)
        errs = stds / np.sqrt(np.maximum(n_folds, 1)) if agg == "sem" else stds
        errs = np.nan_to_num(errs, nan=0.0)

        labels = [f"True\nn={ns[0]}", f"False\nn={ns[1]}", f"No pinch\nn={ns[2]}"]

        axes[i].bar(labels, means, yerr=errs, capsize=3, color=CAT_COLORS[0])
        axes[i].set_title(v)
        if i % 2 == 0:
            axes[i].set_ylabel(ylabel)
        axes[i].set_ylim((0, 1.1))
        clean_axes(axes[i])

    for ax in axes[len(FREE_VARS):]:
        ax.set_axis_off()

    fig.suptitle(f"{title}\n(pinch epochs by variable value; LOSO mean ﷿﷿ {agg.upper()})")
    fig.tight_layout()
    return fig


# _ = plot_free_variable_comparison_loo(
#     results_leave_one_out, _hardlabel_precision, ylabel="Fraction correctly classified",
#     title="Free variable impact on classification precision",
# )

In [ ]:
def _mean_true_proba(metadata, v):
    """Mean predicted probability assigned to the true class per group, for one free variable."""
    def mean_and_n(query):
        sub = metadata.query(query)
        if len(sub) == 0:
            return np.nan, 0
        return sub["proba_true"].mean(), len(sub)

    pinch_true, n_true = mean_and_n(f"{v} == True & EventType == 'PinchLog'")
    pinch_false, n_false = mean_and_n(f"{v} == False & EventType == 'PinchLog'")
    no_pinch, n_no_pinch = mean_and_n("EventType == 'NoPinchLog'")
    return [pinch_true, pinch_false, no_pinch], [n_true, n_false, n_no_pinch]


# _ = plot_free_variable_comparison_loo(
#     results_leave_one_out, _mean_true_proba, ylabel="Mean P(true class)",
#     title="Free variable impact on classification confidence",
# )

## Persist results

In [ ]:
for path in sorted(RESULTS_DIR.glob("*.pkl")):
    with open(path, "rb") as fh:
        cached = pickle.load(fh)
    match = "current" if cached.get("data_version") == DATA_VERSION else "stale"
    print(f"{path.stem:30s} data_version={cached.get('data_version')} ({match})")


In [ ]:
m = robust_epochs_scaled.metadata
m[(m.EventType == 'PinchLog') & m.isChoiceStimuli]